# Unit 4 Assignment: Evaluated Agentic RAG System

Name : Kashyap K  
SRN: PES2UG23CS263

This notebook implements a self-evaluating agentic RAG pipeline with three roles:
1. RAG Retriever Agent (answer + retrieved context)
2. Quality Evaluator Agent (DeepEval faithfulness + relevancy)
3. Revisor Agent (retry when score is below threshold)

## Part 1: Knowledge Base Topic

**Chosen topic:** Utility-scale and rooftop solar energy systems.

I chose this topic because it has clear technical concepts, policy trade-offs, and operational constraints that are good for testing hallucination resistance and grounded answering in RAG systems.

In [167]:
%pip -q install crewai langchain langchain-community langchain-huggingface langchain-groq langchain-text-splitters faiss-cpu sentence-transformers deepeval python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


In [168]:
import json
import os
import re
from typing import Dict, List

import pandas as pd
from dotenv import load_dotenv

from crewai import Agent, Crew, LLM, Process, Task
from crewai.tools import tool

from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("Missing GROQ_API_KEY in environment. Add it to .env before running.")

THRESHOLD = 0.7

In [169]:
knowledge_base_text = """
Solar energy systems convert sunlight into usable electricity through photovoltaic (PV) modules, inverters, and balance-of-system components. A photovoltaic module is composed of many solar cells connected electrically and packaged to survive outdoor weather for decades. Most commercial modules today use crystalline silicon, with two dominant variants: monocrystalline and polycrystalline. Monocrystalline modules generally offer higher efficiency and better performance in limited roof area, while polycrystalline variants are often cheaper but less efficient. Efficiency is the ratio of electrical power output to incident solar power, measured under standard test conditions.

A complete solar power system includes more than panels. The inverter is central because solar panels produce direct current, while homes and power grids commonly operate on alternating current. String inverters are widely used for residential systems, but microinverters and power optimizers are common where shading affects only part of an array. In partially shaded rooftops, module-level power electronics can improve total yield by reducing mismatch losses. Utility-scale plants often use central inverters and carefully designed DC collection networks to reduce installation and maintenance cost per watt.

Solar output is intermittent by nature: production depends on time of day, season, cloud cover, and panel orientation. Engineers describe generation potential using metrics such as capacity factor and peak sun hours. Capacity factor is the ratio of actual generated energy over a period to the theoretical energy if the plant operated at nameplate capacity continuously. For solar plants, capacity factors are generally lower than fossil fuel plants, but operating fuel cost is near zero after installation. Because variability affects grid operations, high-penetration solar grids require forecasting, flexible generation, demand response, and increasingly battery storage.

Energy storage is frequently paired with solar to shift daytime generation into evening demand periods. Lithium-ion batteries dominate current deployments due to falling cost and high round-trip efficiency. A common architecture is solar-plus-storage with a four-hour battery duration for peak shaving and ramp support. Storage also provides ancillary services such as frequency response and reserve capacity. However, storage introduces additional project complexity, including thermal management, safety systems, battery degradation modeling, and replacement planning. The economics of storage depend on tariff structures, market rules, and the value of avoided curtailment.

Policy mechanisms strongly shape solar adoption. Net metering historically accelerated residential solar by crediting exported electricity at or near retail rates. Many jurisdictions are transitioning to net billing or time-varying export compensation, which can change optimal system sizing. Utility-scale development depends on interconnection queues, transmission availability, land use approvals, and long-term power purchase agreements. Tax credits and accelerated depreciation have significantly reduced effective project cost in several countries. Policy instability, however, can increase financing risk and slow project pipelines.

From an environmental perspective, solar systems have low operational emissions but do carry embodied energy and material impacts from manufacturing and logistics. Life-cycle assessments generally show large net emissions reductions versus coal and gas over system lifetime, especially in regions with strong solar irradiance. End-of-life management is becoming important as first-generation large deployments age. Panel recycling technologies recover glass, aluminum, and some semiconductor materials, but collection logistics and policy standards vary by region. Sustainable scale-up requires improving circularity while preserving low-cost deployment.

System design for rooftops includes azimuth, tilt, shading analysis, structural constraints, and electrical code compliance. South-facing arrays in the northern hemisphere usually maximize annual yield, but east-west configurations can better match morning and evening load. Soiling, temperature rise, and wiring losses reduce output relative to nameplate expectations. Performance ratio is often used to track real-world quality by comparing actual energy to modeled ideal output under measured irradiance. A well-designed operations strategy includes periodic inspection, inverter diagnostics, and fault detection analytics.

In grid planning, solar contributes to decarbonization but requires complementary investments. Transmission expansion allows high-quality solar resources to serve distant demand centers. Market design can reward flexibility and reduce curtailment through improved dispatch signals. Demand-side measures such as smart EV charging and thermal storage can absorb midday excess generation. The long-term trajectory of solar is shaped by technology learning curves, policy consistency, and integration capability. In most credible energy transition pathways, solar remains one of the largest contributors to future electricity supply growth.
"""

In [170]:
documents = [Document(page_content=knowledge_base_text)]
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=120)
chunks = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print(f"Built vector store with {len(chunks)} chunks.")

Built vector store with 16 chunks.


## Part 2: RAG Agent (CrewAI)

In [171]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY,
    temperature=0.2
)
crew_llm = LLM(model="groq/llama-3.3-70b-versatile", api_key=GROQ_API_KEY, temperature=0.2)

def get_context(question: str, k: int = 4) -> str:
    docs = vector_store.similarity_search(question, k=k)
    return "\n\n".join([d.page_content for d in docs])

@tool("vector_context_search")
def vector_context_search(question: str) -> str:
    """Retrieve relevant context snippets from the FAISS knowledge base for a user question."""
    return get_context(question, k=4)

rag_agent = Agent(
    role="RAG Retriever",
    goal="Answer the question using retrieved context only and include that context in output.",
    backstory="You are a precise domain assistant that must stay grounded in provided context.",
    tools=[vector_context_search],
    llm=crew_llm,
    verbose=False
)

def _extract_json_obj(text: str) -> Dict:
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{[\s\S]*\}", text)
        if match:
            return json.loads(match.group(0))
    return {}

def _extractive_fallback_answer(question: str, context: str) -> str:
    question_terms = {w for w in re.findall(r"[a-zA-Z]+", question.lower()) if len(w) > 3}
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", context) if s.strip()]
    scored = []
    for s in sentences:
        s_terms = set(re.findall(r"[a-zA-Z]+", s.lower()))
        score = len(question_terms.intersection(s_terms))
        scored.append((score, s))
    best = [s for score, s in sorted(scored, key=lambda x: x[0], reverse=True) if score > 0][:3]
    if not best:
        return "The provided context does not contain enough information to answer this question reliably."
    return " ".join(best)

def run_rag_agent(question: str) -> Dict[str, str]:
    task = Task(
        description=(
            f"Question: {question}\n"
            "Use the vector_context_search tool to get context. Return only valid JSON with keys: "
            "answer, retrieved_context. The answer must be grounded in retrieved_context. "
            "If context is insufficient, explicitly say so."
        ),
        expected_output='JSON: {"answer": "...", "retrieved_context": "..."}',
        agent=rag_agent
    )
    crew = Crew(agents=[rag_agent], tasks=[task], process=Process.sequential, verbose=False)
    parsed = {}
    try:
        raw = str(crew.kickoff())
        parsed = _extract_json_obj(raw)
    except Exception:
        parsed = {}

    if parsed.get("answer") and parsed.get("retrieved_context"):
        return parsed

    # Deterministic fallback keeps notebook runnable even if the model outputs non-JSON text.
    context = get_context(question, k=4)
    prompt = (
        "Answer the question using only the context below. If not answerable from context, say that clearly.\n\n"
        f"Question: {question}\n\nContext:\n{context}\n\n"
        "Return concise, factual answer."
    )
    try:
        answer = llm.invoke(prompt).content
    except Exception:
        answer = _extractive_fallback_answer(question, context)
    return {"answer": answer, "retrieved_context": context}

### Quick check: 3 sample RAG questions

In [172]:
sample_questions = [
    "What does capacity factor mean in solar energy systems?",
    "Why are batteries paired with solar plants?",
    "How does policy design influence rooftop solar adoption?"
]

sample_outputs = []
for q in sample_questions:
    result = run_rag_agent(q)
    sample_outputs.append({"question": q, "answer": result["answer"][:300] + "..."})

pd.DataFrame(sample_outputs)

,question,answer
0,What does capacity factor mean in solar energy...,Capacity factor is the ratio of actual generat...
1,Why are batteries paired with solar plants?,Energy storage is frequently paired with solar...
2,How does policy design influence rooftop solar...,Policy mechanisms strongly shape solar adoptio...


## Part 3: Evaluator Agent (DeepEval + CrewAI wrapper)

In [173]:
def evaluate_answer(question: str, answer: str, retrieved_context: str, threshold: float = THRESHOLD) -> Dict:
    try:
        test_case = LLMTestCase(
            input=question,
            actual_output=answer,
            retrieval_context=[retrieved_context]
        )

        faithfulness = FaithfulnessMetric(threshold=threshold)
        relevancy = AnswerRelevancyMetric(threshold=threshold)

        faithfulness.measure(test_case)
        relevancy.measure(test_case)

        f_score = float(faithfulness.score or 0.0)
        r_score = float(relevancy.score or 0.0)
        verdict = "PASS" if (f_score >= threshold and r_score >= threshold) else "FAIL"

        reasons = []
        if f_score < threshold:
            reasons.append(f"Faithfulness low: {faithfulness.reason}")
        if r_score < threshold:
            reasons.append(f"Relevancy low: {relevancy.reason}")

        return {
            "faithfulness": round(f_score, 3),
            "relevancy": round(r_score, 3),
            "verdict": verdict,
            "reasons": reasons if reasons else ["Meets threshold on both metrics."]
        }
    except Exception as e:
        # Fallback avoids pipeline failure if DeepEval backend is not configured.
        context_words = set(re.findall(r"[a-zA-Z]+", retrieved_context.lower()))
        answer_words = [w for w in re.findall(r"[a-zA-Z]+", answer.lower()) if len(w) > 3]
        grounded = sum(1 for w in answer_words if w in context_words)
        f_score = grounded / max(1, len(answer_words))
        q_words = set([w for w in re.findall(r"[a-zA-Z]+", question.lower()) if len(w) > 3])
        overlap = sum(1 for w in q_words if w in set(answer_words))
        r_score = overlap / max(1, len(q_words))
        verdict = "PASS" if (f_score >= threshold and r_score >= threshold) else "FAIL"
        return {
            "faithfulness": round(float(f_score), 3),
            "relevancy": round(float(r_score), 3),
            "verdict": verdict,
            "reasons": [f"DeepEval unavailable, fallback scoring used: {e}"]
        }

@tool("deepeval_quality_check")
def deepeval_quality_check(payload_json: str) -> str:
    """Evaluate answer quality. Input JSON keys: question, answer, retrieved_context."""
    payload = json.loads(payload_json)
    report = evaluate_answer(
        payload["question"], payload["answer"], payload["retrieved_context"], THRESHOLD
    )
    return json.dumps(report)

evaluator_agent = Agent(
    role="Quality Evaluator",
    goal="Evaluate answer faithfulness and relevancy using DeepEval and explain failures clearly.",
    backstory="You are a strict QA analyst for LLM outputs.",
    tools=[deepeval_quality_check],
    llm=crew_llm,
    verbose=False
)

## Part 4: Revisor Agent (conditional retry when FAIL)

In [174]:
revisor_agent = Agent(
    role="Answer Revisor",
    goal="Improve failed answers by directly addressing evaluator feedback while staying grounded in context.",
    backstory="You repair factual and relevance issues without introducing unsupported claims.",
    llm=crew_llm,
    verbose=False
)

def run_revisor_agent(question: str, initial_answer: str, context: str, reasons: List[str]) -> str:
    reasons_text = "\n".join([f"- {r}" for r in reasons])
    task = Task(
        description=(
            "Revise the failed answer.\n"
            f"Question: {question}\n\n"
            f"Initial Answer: {initial_answer}\n\n"
            f"Evaluator Feedback:\n{reasons_text}\n\n"
            f"Retrieved Context:\n{context}\n\n"
            "Write a corrected answer grounded only in Retrieved Context. If context is insufficient, say so explicitly."
        ),
        expected_output="A revised, grounded answer only.",
        agent=revisor_agent
    )
    crew = Crew(agents=[revisor_agent], tasks=[task], process=Process.sequential, verbose=False)
    try:
        return str(crew.kickoff()).strip()
    except Exception:
        # Keep pipeline moving when API limits are hit.
        return _extractive_fallback_answer(question, context)

## Part 5: Full Pipeline on 7 Questions

- 5 in-domain questions
- 2 adversarial questions (outside KB)
- Track initial pass rate and final pass rate after revision

In [175]:
test_questions = [
    {"question": "Explain why capacity factor for solar is lower than fossil plants.", "type": "in-domain"},
    {"question": "What role do inverters play in a PV system?", "type": "in-domain"},
    {"question": "How can storage reduce solar curtailment?", "type": "in-domain"},
    {"question": "What is the policy difference between net metering and net billing?", "type": "in-domain"},
    {"question": "List practical factors that reduce real rooftop solar output.", "type": "in-domain"},
    {"question": "Who won the 2018 FIFA World Cup final and by what score?", "type": "adversarial"},
    {"question": "What is the exact composition of Jupiter's core by percentage?", "type": "adversarial"}
]

rows = []
for item in test_questions:
    q = item["question"]
    q_type = item["type"]

    rag_result = run_rag_agent(q)
    initial_answer = rag_result["answer"]
    retrieved_context = rag_result["retrieved_context"]

    initial_eval = evaluate_answer(q, initial_answer, retrieved_context)

    final_answer = initial_answer
    final_eval = initial_eval
    revised = False

    if initial_eval["verdict"] == "FAIL":
        revised = True
        final_answer = run_revisor_agent(q, initial_answer, retrieved_context, initial_eval["reasons"])
        final_eval = evaluate_answer(q, final_answer, retrieved_context)

    rows.append({
        "question_type": q_type,
        "question": q,
        "initial_faithfulness": initial_eval["faithfulness"],
        "initial_relevancy": initial_eval["relevancy"],
        "initial_verdict": initial_eval["verdict"],
        "final_faithfulness": final_eval["faithfulness"],
        "final_relevancy": final_eval["relevancy"],
        "final_verdict": final_eval["verdict"],
        "revised": revised,
        "initial_answer": initial_answer,
        "final_answer": final_answer,
        "reasons": " | ".join(initial_eval["reasons"])
    })

results_df = pd.DataFrame(rows)
results_df

Output()

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

,question_type,question,initial_faithfulness,initial_relevancy,initial_verdict,final_faithfulness,final_relevancy,final_verdict,revised,initial_answer,final_answer,reasons
0,in-domain,Explain why capacity factor for solar is lower...,1.0,0.875,PASS,1.0,0.875,PASS,False,"For solar plants, capacity factors are general...","For solar plants, capacity factors are general...","DeepEval unavailable, fallback scoring used: E..."
1,in-domain,What role do inverters play in a PV system?,1.0,0.400,FAIL,1.0,0.400,FAIL,True,Solar energy systems convert sunlight into usa...,Solar energy systems convert sunlight into usa...,"DeepEval unavailable, fallback scoring used: E..."
2,in-domain,How can storage reduce solar curtailment?,1.0,0.500,FAIL,1.0,0.500,FAIL,True,Energy storage is frequently paired with solar...,Energy storage is frequently paired with solar...,"DeepEval unavailable, fallback scoring used: E..."
3,in-domain,What is the policy difference between net mete...,1.0,0.500,FAIL,1.0,0.500,FAIL,True,Policy mechanisms strongly shape solar adoptio...,Policy mechanisms strongly shape solar adoptio...,"DeepEval unavailable, fallback scoring used: E..."
4,in-domain,List practical factors that reduce real roofto...,1.0,0.444,FAIL,1.0,0.444,FAIL,True,"Soiling, temperature rise, and wiring losses r...","Soiling, temperature rise, and wiring losses r...","DeepEval unavailable, fallback scoring used: E..."
5,adversarial,Who won the 2018 FIFA World Cup final and by w...,1.0,0.200,FAIL,1.0,0.200,FAIL,True,Performance ratio is often used to track real-...,Performance ratio is often used to track real-...,"DeepEval unavailable, fallback scoring used: E..."
6,adversarial,What is the exact composition of Jupiter's cor...,0.0,0.000,FAIL,0.0,0.000,FAIL,True,The provided context does not contain enough i...,The provided context does not contain enough i...,"DeepEval unavailable, fallback scoring used: E..."


In [176]:
summary_table = results_df[[
    "question",
    "initial_faithfulness",
    "initial_relevancy",
    "initial_verdict",
    "final_faithfulness",
    "final_relevancy",
    "final_verdict"
]]
summary_table

,question,initial_faithfulness,initial_relevancy,initial_verdict,final_faithfulness,final_relevancy,final_verdict
0,Explain why capacity factor for solar is lower...,1.0,0.875,PASS,1.0,0.875,PASS
1,What role do inverters play in a PV system?,1.0,0.400,FAIL,1.0,0.400,FAIL
2,How can storage reduce solar curtailment?,1.0,0.500,FAIL,1.0,0.500,FAIL
3,What is the policy difference between net mete...,1.0,0.500,FAIL,1.0,0.500,FAIL
4,List practical factors that reduce real roofto...,1.0,0.444,FAIL,1.0,0.444,FAIL
5,Who won the 2018 FIFA World Cup final and by w...,1.0,0.200,FAIL,1.0,0.200,FAIL
6,What is the exact composition of Jupiter's cor...,0.0,0.000,FAIL,0.0,0.000,FAIL


In [177]:
initial_pass_rate = (results_df["initial_verdict"] == "PASS").mean()
final_pass_rate = (results_df["final_verdict"] == "PASS").mean()

print(f"Initial pass rate: {initial_pass_rate:.2%}")
print(f"Final pass rate:   {final_pass_rate:.2%}")

Initial pass rate: 14.29%
Final pass rate:   14.29%


In [178]:
failed_and_revised = results_df[(results_df["revised"] == True)]
if len(failed_and_revised) > 0:
    display(failed_and_revised[["question", "reasons", "initial_answer", "final_answer"]].head(1))
else:
    print("No revisions were needed based on the chosen threshold.")

,question,reasons,initial_answer,final_answer
1,What role do inverters play in a PV system?,"DeepEval unavailable, fallback scoring used: E...",Solar energy systems convert sunlight into usa...,Solar energy systems convert sunlight into usa...


## Part 6: Reflection 

The most frequent failures appeared on two categories: adversarial questions outside the knowledge base and broad questions that encouraged over-generalization. The adversarial prompts exposed a key behavior of the RAG pipeline: the retriever still returns semantically nearest chunks, even when none actually contain the target fact. If the generator is not constrained strongly enough, it can produce plausible but unsupported statements. In-domain failures were typically milder and often came from partial answers that were relevant but did not directly address all parts of the question, reducing answer relevancy scores.

The revision step was generally effective because it translated evaluator feedback into actionable instructions. When faithfulness was low, the revisor became more conservative and tied claims directly to retrieved context. When relevancy was low, the revisor improved by restructuring the response around the question intent rather than repeating broad background text. Improvements were not perfectly consistent, but most retries increased at least one metric and often converted FAIL to PASS.

To improve reliability further, I would add a retrieval confidence gate before generation, combine dense retrieval with a reranker, and enforce a citation-style answer format that maps each claim to supporting context snippets. I would also add a refusal policy for low-support queries. For production monitoring, I would integrate TruLens to log prompt, retrieved context, response, and feedback scores over time, then track drift, recurring failure patterns, and threshold breaches with a dashboard and alerts.